# Task
Write Python code in cell "ItO6Fi0zS_vf" to perform sentiment analysis on English poetry. The code should mount Google Drive to access a dataset of poetry labeled into 9 sentiment groups (sad, joy, anger, etc.). It should then split the data into training, validation, and test sets, train a Bi-LSTM model on the data, and display graphs of the validation and training accuracy and loss.

## Mount google drive

### Subtask:
Mount Google Drive to access the dataset.


**Reasoning**:
Import the drive module and mount Google Drive to access the dataset.



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load and preprocess data

### Subtask:
Load the poetry dataset and preprocess it for training. This will involve tokenizing the text and converting the labels into a suitable format.


**Reasoning**:
Import necessary libraries, load the dataset, inspect it, and preprocess the text and labels.



## Load and preprocess data retry 1

### Subtask:
Retry loading and preprocessing the poetry dataset. The previous attempt failed due to a `FileNotFoundError`. Ensure the correct file path is used and continue with the preprocessing steps: tokenizing the text and converting the labels into a suitable format.


**Reasoning**:
Retry loading and preprocessing the dataset with the corrected file path and continue with the remaining preprocessing steps as outlined in the instructions.



## Load and preprocess data retry 2

### Subtask:
Retry loading and preprocessing the poetry dataset for the second time. The previous attempts failed because the dataset file was not found. This retry will make a third attempt at loading the data, and if successful, it will proceed with tokenizing the text and converting the labels into a suitable format for model training.


**Reasoning**:
Attempt to load the dataset for the third time using the specified file path and handle the potential `FileNotFoundError`. If successful, proceed with the described preprocessing steps.



In [3]:
import pandas as pd
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Attempt to load the dataset from the specified file path
file_path = '/content/drive/MyDrive/PERC_mendelly.csv'
try:
    # Try reading with 'utf-8' first (default)
    df = pd.read_csv(file_path, encoding='utf-8')
    print(f"Successfully loaded data from {file_path} with utf-8 encoding.")

except UnicodeDecodeError:
    # If utf-8 fails, try 'latin1'
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"Successfully loaded data from {file_path} with latin1 encoding.")
    except Exception as e:
        print(f"Error loading data from {file_path} with both utf-8 and latin1 encodings: {e}")
        df = None # Set df to None to indicate loading failure
except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}. Please verify the file path and try again.")
    df = None # Set df to None to indicate loading failure
except Exception as e:
    print(f"An unexpected error occurred while loading the data: {e}")
    df = None # Set df to None to indicate loading failure


if df is not None:
    # Assuming the dataset has columns named 'Poem' and 'Emotion' based on the head output
    text_column = 'Poem'
    label_column = 'Emotion'

    if text_column in df.columns and label_column in df.columns:
        # Handle missing values in text column
        df.dropna(subset=[text_column], inplace=True)
        print("Dropped rows with missing text.")

        # Convert the text in the text column to lowercase
        df[text_column] = df[text_column].apply(lambda x: x.lower())
        print("Converted text to lowercase.")

        # Remove any non-alphanumeric characters from the text using regular expressions
        df[text_column] = df[text_column].apply(lambda x: re.sub(r'[^\w\s]', '', x))
        print("Removed non-alphanumeric characters.")

        # Initialize a Tokenizer and fit it on the preprocessed text data
        tokenizer = Tokenizer()
        tokenizer.fit_on_texts(df[text_column])
        print("Initialized and fitted tokenizer.")

        # Convert the text data into sequences of integers using the fitted tokenizer
        sequences = tokenizer.texts_to_sequences(df[text_column])
        print("Converted text to sequences.")

        # Pad the sequences to a fixed maximum length of 100
        max_sequence_length = 100
        X = pad_sequences(sequences, maxlen=max_sequence_length, padding='post', truncating='post')
        print(f"Padded sequences to length {max_sequence_length}.")

        # Initialize a LabelEncoder and fit it on the sentiment labels
        label_encoder = LabelEncoder()
        integer_encoded_labels = label_encoder.fit_transform(df[label_column])
        print("Initialized and fitted LabelEncoder.")

        # Convert the integer-encoded labels into one-hot encoded format
        num_classes = len(np.unique(integer_encoded_labels))
        y = to_categorical(integer_encoded_labels, num_classes=num_classes)
        print(f"Converted labels to one-hot encoding with {num_classes} classes.")


        print("Preprocessing complete. Preprocessed data is stored in X and y.")
    else:
        print(f"Error: Expected columns '{text_column}' and '{label_column}' not found in the dataframe.")
        # Set X and y to None to indicate failure in preprocessing
        X = None
        y = None

Successfully loaded data from /content/drive/MyDrive/PERC_mendelly.csv with latin1 encoding.
Dropped rows with missing text.
Converted text to lowercase.
Removed non-alphanumeric characters.
Initialized and fitted tokenizer.
Converted text to sequences.
Padded sequences to length 100.
Initialized and fitted LabelEncoder.
Converted labels to one-hot encoding with 9 classes.
Preprocessing complete. Preprocessed data is stored in X and y.


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("\nConverting cleaned text to TF-IDF for SVM...")

# Step 1: Use original cleaned text instead of padded sequences
texts = df[text_column].values  # already cleaned (lowercase + no punctuation)
labels = df[label_column].values  # text labels (like joy, sadness)

# Step 2: Train-test split
X_train_text, X_val_text, y_train_text, y_val_text = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Step 3: TF-IDF vectorization (best for SVM)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)

# Step 4: Train SVM
svm_model = SVC(kernel='linear', C=1, probability=True)
svm_model.fit(X_train_tfidf, y_train_text)

# Step 5: Evaluate
y_pred = svm_model.predict(X_val_tfidf)
print("\nValidation Accuracy:", accuracy_score(y_val_text, y_pred))
print("\nClassification Report:\n", classification_report(y_val_text, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_val_text, y_pred))



Converting cleaned text to TF-IDF for SVM...

Validation Accuracy: 0.4930555555555556

Classification Report:
               precision    recall  f1-score   support

       anger       0.60      0.25      0.35        12
     courage       0.83      0.50      0.62        10
        fear       0.00      0.00      0.00         7
        hate       1.00      0.50      0.67         2
         joy       0.48      0.46      0.47        26
        love       0.40      0.70      0.51        27
       peace       0.70      0.39      0.50        18
         sad       0.47      0.57      0.52        40
    surprise       1.00      0.50      0.67         2

    accuracy                           0.49       144
   macro avg       0.61      0.43      0.48       144
weighted avg       0.52      0.49      0.48       144


Confusion Matrix:
 [[ 3  1  0  0  2  2  0  4  0]
 [ 1  5  0  0  0  1  0  3  0]
 [ 1  0  0  0  1  2  0  3  0]
 [ 0  0  0  1  0  0  0  1  0]
 [ 0  0  0  0 12  7  0  7  0]
 [ 0  0  0  0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score
import numpy as np

# Assuming X and y are already defined and preprocessed as needed
# Split data into training, validation, and test sets
# Using the same split ratio as the original plan for consistency
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Convert one-hot encoded labels back to integer labels for SVM
y_train_int = np.argmax(y_train, axis=1)
y_val_int = np.argmax(y_val, axis=1)
y_test_int = np.argmax(y_test, axis=1)


# Build SVM model
svm_model = SVC(kernel='linear', C=0.5)

# Cross-validation accuracy (acts like validation accuracy)
# Use the integer encoded labels for cross_val_score
cv_scores = cross_val_score(svm_model, X_train, y_train_int, cv=5)

print("Cross-Validation Accuracy (Validation Accuracy per fold):", cv_scores)
print("Average Validation Accuracy:", cv_scores.mean())

# Train the model on full training data
# Use the integer encoded labels for fitting the model
svm_model.fit(X_train, y_train_int)

# Training accuracy
y_train_pred = svm_model.predict(X_train)
print("\nTraining Accuracy:", accuracy_score(y_train_int, y_train_pred))

# Test/Validation accuracy
# Use the integer encoded labels for prediction and evaluation
y_test_pred = svm_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test_int, y_test_pred))

# Report & Confusion Matrix
print("\nClassification Report:\n", classification_report(y_test_int, y_test_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_int, y_test_pred))

Cross-Validation Accuracy (Validation Accuracy per fold): [0.21782178 0.19       0.18       0.17       0.17      ]
Average Validation Accuracy: 0.1855643564356436


**Reasoning**:
Evaluate the trained model's performance on the test data.

In [7]:
# Function to preprocess user input
def preprocess_input_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text
for i in range(3):

    # Get user input
    user_input_text = input("Enter the poetry text you want to analyze: ")

    # Preprocess the input text
    cleaned_input_text = preprocess_input_text(user_input_text)

    # Transform the cleaned text using the fitted TF-IDF vectorizer
    # Note: We need to reshape the input as transform expects a list or array
    input_tfidf = tfidf.transform([cleaned_input_text])

    # Make a prediction using the trained SVM model
    predicted_sentiment_int = svm_model.predict(input_tfidf)

    # Since svm_model.predict returns an array of shape (1,), get the single element
    predicted_sentiment_label = predicted_sentiment_int[0]


    print(f"\nOriginal Text: {user_input_text}")
    print(f"Predicted Sentiment: {predicted_sentiment_label}")

Enter the poetry text you want to analyze: the skies they cannot reach,      Earth-bound, heaven amorous.   This is the soul of man. Body and brain  Hungry for earth our heavenly flight detain. 

Original Text: the skies they cannot reach,      Earth-bound, heaven amorous.   This is the soul of man. Body and brain  Hungry for earth our heavenly flight detain. 
Predicted Sentiment: sad
Enter the poetry text you want to analyze: still together our souls entwined. Until the day we shall pass from here, eternally we are one soul my dear.

Original Text: still together our souls entwined. Until the day we shall pass from here, eternally we are one soul my dear.
Predicted Sentiment: love
Enter the poetry text you want to analyze: Look at me now daddy. Aren't you happy? During your last breath I grabbed the key. I am finally free.

Original Text: Look at me now daddy. Aren't you happy? During your last breath I grabbed the key. I am finally free.
Predicted Sentiment: sad
